# 01 — Problem Definition and Experimental Design

**Maps to:** `report/Chapters/Task1.tex` (label `Task1`).  
**Tickets:** TICKET-01 (benchmark choice), TICKET-02 (formal formulation), TICKET-04 (loader + distance matrix).

Code that backs this notebook lives in `src/ga/instance.py`.

## 1.1 Benchmark TSP Instance — TICKET-01
Maps to `T1:Benchmark`.

Document: name, dimension, edge-weight type, known optimal tour length, source citation. Justify the choice (size ≥70 ≤150, instance class).

Candidates: `eil101`, `kroA100`, `ch130`, `bier127`. Final pick → write into `Task1.tex` §`T1:Benchmark` and into the README dataset entry.

In [2]:
# TICKET-04 — exercise src/ga/instance.py here.
# from ga.instance import load_tsp
# coords, dist_matrix = load_tsp('../data/<chosen>.tsp')
# print(coords.shape, dist_matrix.shape)

In [ ]:
# Scatter of cities — sanity check the loader. Save figure to results/figures/ for Task1.

## 1.2 Formal Mathematical Formulation — TICKET-02
Maps to `T1:Formulation`.

Pure-prose ticket. Draft the formulation here as Markdown / LaTeX-in-Markdown, then promote into `Task1.tex` §`T1:Formulation`:
- decision variables $x_{ij} \in \{0,1\}$;
- objective: minimise $\sum_{i,j} c_{ij} x_{ij}$;
- constraints (visit-once, closed tour, sub-tour elimination);
- properties relevant to evolutionary optimisation.

```
The Traveling Salesman Problem (TSP) is a combinatorial optimization problem where the objective is to find the shortest possible route that visits every city exactly once and returns to the starting city.

In this project, the selected benchmark instance is **kroA100** from TSPLIB95. This instance contains **100 cities**, where each city is represented by two-dimensional coordinates. The distance between cities is calculated using the **EUC_2D** Euclidean distance metric.

The TSP can be represented as a complete weighted graph:
```


The TSP can be represented as a complete weighted graph:

$$
G = (V, E)
$$

where:

- $V = \{1,2,\ldots,n\}$ is the set of cities
- $E$ is the set of edges connecting the cities
- $n = 100$ for the kroA100 instance
- $d_{ij}$ represents the distance between city $i$ and city $j$

### Decision Variables



The decision variable is defined as:

$$
x_{ij} =
\begin{cases}
1, & \text{if the tour travels directly from city } i \text{ to city } j \\
0, & \text{otherwise}
\end{cases}
$$

where:

$$
i,j \in V, \quad i \neq j
$$

This means that $x_{ij}$ shows whether the path from city $i$ to city $j$ is selected in the final tour.

Where $i,j \in V$ means that both cities $i$ and $j$ belong to the set of cities, and $i \neq j$ means that the tour cannot travel from a city back to the same city directly.

### Objective Function


The objective of the TSP is to minimize the total distance travelled by the salesman.

$$
\min Z = \sum_{i=1}^{n} \sum_{\substack{j=1 \\ j \neq i}}^{n} d_{ij}x_{ij}
$$

where:

- $Z$ is the total tour distance
- $d_{ij}$ is the distance from city $i$ to city $j$
- $x_{ij}$ indicates whether the edge from city $i$ to city $j$ is selected
- $n = 100$ for the kroA100 instance

The goal is to find the tour with the smallest possible value of $Z$.

### Constraints

#### 1. Each city must be entered exactly once

$$
\sum_{i=1, i \neq j}^{n} x_{ij} = 1,
\quad \forall j \in V
$$

This constraint ensures that every city is visited exactly once.

#### 2. Each city must be left exactly once

$$
\sum_{\substack{j=1 \\ j \neq i}}^{n} x_{ij} = 1,
\quad \forall i \in V
$$

This constraint ensures that every city has exactly one outgoing route. In other words, after visiting a city, the tour must leave that city and continue to exactly one other city.

#### 3. The tour must return to the starting city

The TSP requires a closed tour. This means the salesman must return to the starting city after visiting all other cities.

This condition is achieved because every city has exactly one incoming edge and one outgoing edge.


#### 4. Subtour Elimination Constraint

The previous constraints may still produce smaller disconnected cycles known as subtours. Therefore, subtour elimination constraints are needed:

$$
\sum_{i \in S} \sum_{\substack{j \in S \\ j \neq i}} x_{ij} \leq |S| - 1,
\quad \forall S \subset V, \quad 2 \leq |S| \leq n-1
$$

This constraint prevents the formation of smaller cycles that do not include all cities.

This constraint makes sure that the solution forms one complete tour visiting all cities, instead of forming two or more small separate loops.

#### 5. Binary Constraint

$$
x_{ij} \in \{0,1\},
\quad \forall i,j \in V, \quad i \neq j
$$

This means that each edge is either selected or not selected.

If $x_{ij} = 1$, the path from city $i$ to city $j$ is included in the tour.

If $x_{ij} = 0$, the path from city $i$ to city $j$ is not included in the tour.

The value of xij can only be 0 or 1. It cannot be 0.5, 2, or any other value.

### Properties Relevant to Evolutionary Optimization

The TSP is suitable for Genetic Algorithm optimization because it has a very large search space. A solution to the TSP can be represented as a permutation of cities.

For $n$ cities, the number of possible unique tours is:

$$
\frac{(n-1)!}{2}
$$

```
For the kroA100 instance with 100 cities, the number of possible tours is extremely large. Because of this, checking every possible solution using exhaustive search is not practical.
```

In a Genetic Algorithm, each chromosome represents one possible TSP tour. For example, a chromosome may be written as:

$$
[1, 5, 12, 8, 20, \ldots, 100]
$$

This represents the visiting order of the cities. The return to the starting city is assumed when calculating the total tour distance.

```
This represents the order in which the cities are visited. A valid chromosome must include each city exactly once before returning to the starting city.

However, crossover and mutation operations may sometimes produce infeasible solutions. For example, a chromosome may contain repeated cities or missing cities. Therefore, repair mechanisms are important to convert infeasible chromosomes back into valid TSP tours.

The kroA100 instance is also useful for Genetic Algorithm analysis because it uses Euclidean distance. This means nearby cities may form useful partial routes, also known as building blocks. These building blocks can be preserved and combined through crossover and mutation to produce better solutions over generations.

Overall, the TSP is appropriate for evolutionary optimization because it has a large search space, permutation-based solutions, and strong dependency between the order of cities and the final tour distance.
```

## 1.3 Experimental-Setup Justification — feeds TICKET-15/16
Maps to `T1:Setup`.

Decide and justify:
- number of independent runs (target ≥30);
- termination (max generations / convergence threshold);
- parameter ranges for the sweep (population size, crossover rate, mutation rate, selection method).

These decisions become the default config dict consumed by `experiments/run.py` and `experiments/sweep.py`.